In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import matplotlib as mpl
from matplotlib.colors import LogNorm, Normalize
from mpl_toolkits.axes_grid1.inset_locator import zoomed_inset_axes
from mpl_toolkits.axes_grid1.inset_locator import mark_inset
import matplotlib.colors as mcolors

from astropy.table import Table, join, QTable, vstack
import astropy.table as table
from astropy.io import fits, ascii
import astropy.units as u
from astropy.nddata import NDData, Cutout2D
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
from astropy.wcs.utils import pixel_to_skycoord, skycoord_to_pixel
import astropy

import sys
import math
import pyneb as pn
from multiprocessing import Pool
import multiprocessing as mp
from itertools import compress
import random

import scipy.constants
import scipy.stats as stats
from scipy.optimize import curve_fit
from scipy.integrate import quad
from scipy.interpolate import griddata
from scipy.stats import spearmanr, pearsonr

from orcs.process import SpectralCube
from orb.fit import fit_lines_in_spectrum
from orb.utils.spectrum import corr2theta, amp_ratio_from_flux_ratio
from orb.core import Lines
import gvar
import orb

from reproject import reproject_interp
import reproject
from regions import PixCoord

import pylab as pl
from matplotlib import cm

import extinction
from extinction import apply, remove

from photutils.detection import DAOStarFinder

import aplpy
import seaborn as sns

from skimage import filters

import statsmodels.api as sm

sys.path.append("/home/habjan/SITELLE/sitelle_metallicities")
import analysis_functions as af


### Output path for figures

In [ ]:
path_out = '/home/habjan/SITELLE/sitelle_metallicities/figures_1/'

### Font style

In [ ]:
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "dejavuserif",
    "text.usetex": False,
})


# Figure 1: NGC 628 [O II] flux map with H II region mask overlaid

In [ ]:
products_data_path = '/home/habjan/SITELLE/data/data_products'
inter_data_path = '/home/habjan/SITELLE/data/data_raw_intermediate'

sit_0628 = fits.open(products_data_path + f"/NGC0628_SITELLE.fits")
OII_0628_mp = fits.open(products_data_path + f"/NGC0628_OII_map_mp.fits")[0]
OII_0628 = fits.open(products_data_path + f"/NGC0628_OII_map.fits")[0]
mask_0628 = fits.open(inter_data_path + f"/NGC0628_nebulae_mask_V2.fits")[0]
infile = open(inter_data_path + f'/NGC0628_physdata_MUSE+SITELLE.fits','rb')
data_0628 = Table.read(infile)
infile = inter_data_path + f"/NGC0628_cube.hdf5"
cube_0628 = SpectralCube(infile)
sit_0628_mp = fits.open(products_data_path + f"/NGC0628_SITELLE_mp.fits")

sit_2835 = fits.open(products_data_path + f"/NGC2835_SITELLE.fits")
#OII_2835_mp = fits.open(f"/NGC2835_OII_map_mp.fits")[0]
OII_2835 = fits.open(products_data_path + f"/NGC2835_OII_map.fits")[0]
mask_2835 = fits.open(inter_data_path + f"/NGC2835_nebulae_mask_V2.fits")[0]
infile = open(inter_data_path + f'/NGC2835_physdata_MUSE+SITELLE.fits','rb')
data_2835 = Table.read(infile)
infile = inter_data_path + f"/NGC2835_cube.hdf5"
cube_2835 = SpectralCube(infile)

sit_3351 = fits.open(products_data_path + f"/NGC3351_SITELLE.fits")
#OII_3351_mp = fits.open(f"/NGC3351_OII_map_mp.fits")[0]
OII_3351 = fits.open(products_data_path + f"/NGC3351_OII_map.fits")[0]
mask_3351 = fits.open(inter_data_path + f"/NGC3351_nebulae_mask_V2.fits")[0]
infile = open(inter_data_path + f'/NGC3351_physdata_MUSE+SITELLE.fits','rb')
data_3351 = Table.read(infile)
infile = inter_data_path + f"/NGC3351_cube.hdf5"
cube_3351 = SpectralCube(infile)

sit_4535 = fits.open(products_data_path + f"/NGC4535_SITELLE.fits")
#OII_4535_mp = fits.open(f"/NGC4535_OII_map_mp.fits")[0]
OII_4535 = fits.open(products_data_path + f"/NGC4535_OII_map.fits")[0]
mask_4535 = fits.open(inter_data_path + f"/NGC4535_nebulae_mask_V2.fits")[0]
infile = open(inter_data_path + f'/NGC4535_physdata_MUSE+SITELLE.fits','rb')
data_4535 = Table.read(infile)
infile = inter_data_path + f"/NGC4535_cube.hdf5"
cube_4535 = SpectralCube(infile)

sit_3627 = fits.open(products_data_path + f"/NGC3627_SITELLE.fits")
#OII_4535_mp = fits.open(f"/NGC4535_OII_map_mp.fits")[0]
#OII_3627 = fits.open(f"/NGC3627_OII_map.fits")[0]
mask_3627 = fits.open(inter_data_path + f"/NGC3627_nebulae_mask_V2.fits")[0]
infile = open(inter_data_path + f'/NGC3627_physdata_MUSE+SITELLE.fits','rb')
data_3627 = Table.read(infile)
infile = inter_data_path + f"/NGC3627_cube.hdf5"
cube_3627 = SpectralCube(infile)

table_dic = {1: data_0628, 2:data_2835, 3:data_3351, 4:data_4535, 5:data_3627}

### Make a velocity map

In [ ]:
snr = 5
high_vel_ind = np.where(data_0628['OII3727_FLUX_CORR'] / data_0628['OII3727_FLUX_CORR_ERR'] > snr)[0]

vel_mask = np.zeros(mask_0628.data.shape)
for i in high_vel_ind:
    vel_mask[mask_0628.data == i] = data_0628['OII3727_VEL'][i]

x, y = np.where(vel_mask != 0)
x_grid, y_grid = np.meshgrid(np.arange(vel_mask.shape[0]), np.arange(vel_mask.shape[1]))
vel_smoothed = griddata(points = (x, y), values = vel_mask[np.where(vel_mask != 0)], xi=(x_grid, y_grid), method='nearest')
vel_map_0628 = fits.PrimaryHDU(data=vel_smoothed, header=mask_0628.header)

### Make a smoothed [O II] map

In [ ]:
wave3729 = 3728.815
c = 299792
galvel = 651

up_down = 15

test = fits.open(inter_data_path + f"/NGC0628_SITELLE_mp.fits")

### Go from Velocities to wavenumber
red3729 = 1/((wave3729*(vel_map_0628.data+galvel)/(c) + wave3729) * 10**-8)
### Make a 3d boolean above and below the peak of [OII]3727
wave_bool = (red3729 + up_down > sit_0628[1].data[:, None, None]) & (red3729 - up_down < sit_0628[1].data[:, None, None])
### make a masked array of where [OII]3727 is at each point in the cube
#cube_masked = np.ma.masked_array(sit_0628_mp[0].data, ~wave_bool)
cube_masked = np.ma.masked_array(test[0].data, ~wave_bool)
### sum over all wavelength channels
oii_map_sum = np.nansum(cube_masked, axis=0)
### Smooth the OII map 
oii_smoothed = filters.gaussian(oii_map_sum, sigma=1.5)
### Save as a fits file
oii_map = fits.PrimaryHDU(data=oii_smoothed, header=mask_0628.header)

### Make the plot

In [ ]:
#reg1, reg2 = 184, 311

hii_mask = np.copy(mask_0628.data)
#hii_mask[hii_mask == reg1] = 10**4
#hii_mask[hii_mask == reg2] = 10**4
#hii_mask[hii_mask < 10**3.9] = -1

hii_mask[hii_mask > 0] = 10**4

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(projection=WCS(mask_0628.header))

ax.imshow(oii_map.data, cmap='Greys', vmin=np.nanquantile(oii_map.data, 0.1), vmax=np.nanquantile(oii_map.data, 0.985))
ax.contour(hii_mask, colors='red', alpha=0.17)
ax.update({'xlabel': 'R.A.', 'ylabel': 'Dec'})
ax.grid(False)

axins = zoomed_inset_axes(ax, 2.5, loc=4)
axins.imshow(oii_map.data, cmap='Greys', vmin=np.nanquantile(oii_map.data, 0.1), vmax=np.nanquantile(oii_map.data, 0.985))
axins.contour(hii_mask, colors='red', alpha=0.17)

lim_dic = {'NGC4535':[475, 575, 750, 850], 'NGC3351':[325, 450, 735, 860], 'NGC2835':[600, 700, 275, 375] ,'NGC0628':[635, 760, 370, 495]}

galaxy = 'NGC0628'

axins.set_xlim(lim_dic[galaxy][0], lim_dic[galaxy][1])
axins.set_ylim(lim_dic[galaxy][2], lim_dic[galaxy][3])

plt.xticks(visible=False)
plt.yticks(visible=False)
plt.grid(False)
mark_inset(ax, axins, loc1=3, loc2=1, fc="none", ec="0.0")
plt.draw()
plt.show()

fig.savefig(path_out+'OII_map.png',dpi=300,bbox_inches='tight')

# Figure 2: [O II]λ3727 spectral fits at varying SNR

In [ ]:
galaxynum = 4
galdic = {1:'NGC4254', 2:'NGC4535', 3:'NGC3351', 4:'NGC2835', 5:'NGC0628', 6:'NGC3627'}  #There is no SITELLE data for NGC 4254, NGC 2835 has the best data 
galaxy = galdic[galaxynum]

galveldic = {'NGC4254': 2388 , 'NGC4535': 1954  , 'NGC3351': 775, 'NGC2835': 867, 'NGC0628':651, 'NGC3627':715}
galvel = galveldic[galaxy]

hdul = fits.open(f'/home/habjan/SITELLE/sandbox_notebooks/pdf_fit_plotting/{galaxy}_spectra_fitting.fits')

infile = open(inter_data_path + f'/{galaxy}_refit+SITELLEfits_data.fits','rb')
data = Table.read(infile)

sitlam = hdul[0].data
sitspec = hdul[1].data

fit3727 = hdul[2].data
snr3727 = hdul[3].data

wave_surr = 50
flux_bottom = 10**-0.6
flux_top = 1.1
x_min = 3710
x_max = 3770

fig, (ax1, ax2, ax3)  = plt.subplots(1, 3)#, sharey='row')
fig.set_figheight(5)
fig.set_figwidth(14)

### Plot 1

reg = np.where(np.nanmax(snr3727) == snr3727)[0][0]

print(f'Region {reg} with SNR {snr3727[reg]}')

inspectrum = sitspec[reg]
inspectrum[np.isnan(inspectrum)] = 0

wave3729 = 3728.815
c = 299792
red3729 = 1/((wave3729*(data[reg]['HA6562_VEL']+galvel)/(c) + wave3729) * 10**-8)

n1 = [np.where(sitlam > red3729 - 450)[0][0],np.where(sitlam > red3729 - 175)[0][0]]
n2 = [np.where(sitlam > red3729 + 175)[0][0], np.where(sitlam > red3729 + 700)[0][0]]

fit_lam = np.concatenate([sitlam[n2[0]:n2[1]], sitlam[n1[0]:n1[1]]])
fit_spec = np.concatenate([inspectrum[n2[0]:n2[1]], inspectrum[n1[0]:n1[1]]])
s_fit = np.polyfit(fit_lam, fit_spec, 1)
sfit_func = np.poly1d(s_fit)

noisestd = (np.std(inspectrum[n1[0]:n1[1]]) + np.std(inspectrum[n2[0]:n2[1]])) / 2
inspectrum = inspectrum - sfit_func(sitlam)

flux = np.array(data['OII3727_FLUX_CORR'])[reg]

ax1.plot(1/(10**-8*sitlam), inspectrum, c='k')#, label='spectrum')
ax1.plot(1/(10**-8*sitlam), fit3727[reg], c='m', label=f'{galaxy} Region {reg}')
ax1.legend(fontsize=12, loc='upper right')


#ax1.set_title(f'Region {reg} | SNR: {round(snr3727[reg], 2)}')
ax1.set_xlim(x_min, x_max)
max_spec = np.nanmax(sitspec[reg][(sitlam > 1/(3850*10**-8)) & (sitlam < 1/(3650*10**-8))])
min_spec = np.nanmin(sitspec[reg][(sitlam > 1/(3850*10**-8)) & (sitlam < 1/(3650*10**-8))])
ax1.set_ylim(min_spec*flux_bottom * 10, max_spec*flux_top)
ax1.fill_between(1/(10**-8*sitlam)[np.where(sitlam > red3729 - 450)[0][0]:np.where(sitlam > red3729 - 175)[0][0]],-10**6, 10**10, color='purple', alpha = 0.15)
ax1.fill_between(1/(10**-8*sitlam)[np.where(sitlam > red3729 + 175)[0][0]:np.where(sitlam > red3729 + 700)[0][0]],-10**6, 10**10, color='purple', alpha = 0.15)

### Plot 2

reg = np.where((snr3727 > 10) & (snr3727 < 15))[0][0]
print(f'Region {reg} with SNR {snr3727[reg]}')

inspectrum = sitspec[reg]
inspectrum[np.isnan(inspectrum)] = 0

wave3729 = 3728.815
c = 299792
red3729 = 1/((wave3729*(data[reg]['HA6562_VEL']+galvel)/(c) + wave3729) * 10**-8)

n1 = [np.where(sitlam > red3729 - 450)[0][0],np.where(sitlam > red3729 - 175)[0][0]]
n2 = [np.where(sitlam > red3729 + 175)[0][0], np.where(sitlam > red3729 + 700)[0][0]]

fit_lam = np.concatenate([sitlam[n2[0]:n2[1]], sitlam[n1[0]:n1[1]]])
fit_spec = np.concatenate([inspectrum[n2[0]:n2[1]], inspectrum[n1[0]:n1[1]]])
s_fit = np.polyfit(fit_lam, fit_spec, 1)
sfit_func = np.poly1d(s_fit)

noisestd = (np.std(inspectrum[n1[0]:n1[1]]) + np.std(inspectrum[n2[0]:n2[1]])) / 2
inspectrum = inspectrum - sfit_func(sitlam)

flux = np.array(data['OII3727_FLUX_CORR'])[reg]

ax2.plot(1/(10**-8*sitlam), inspectrum, c='k')#, label='spectrum')
ax2.plot(1/(10**-8*sitlam), fit3727[reg], c='red', label=f'{galaxy} Region {reg}')

#ax2.set_title(f'Region {reg} | SNR: {round(snr3727[reg], 2)}')
ax2.set_xlim(x_min, x_max)
max_spec = np.nanmax(sitspec[reg][(sitlam > 1/(3850*10**-8)) & (sitlam < 1/(3650*10**-8))])
min_spec = np.nanmin(sitspec[reg][(sitlam > 1/(3850*10**-8)) & (sitlam < 1/(3650*10**-8))])
ax2.set_ylim(min_spec*flux_bottom *3, max_spec*flux_top)
ax2.legend(fontsize=12, loc='upper right')

ax2.fill_between(1/(10**-8*sitlam)[np.where(sitlam > red3729 - 450)[0][0]:np.where(sitlam > red3729 - 175)[0][0]],-10**6, 10**10, color='purple', alpha = 0.15)
ax2.fill_between(1/(10**-8*sitlam)[np.where(sitlam > red3729 + 175)[0][0]:np.where(sitlam > red3729 + 700)[0][0]],-10**6, 10**10, color='purple', alpha = 0.15)

### Plot 3

reg = np.where((snr3727 > 3) & (snr3727 < 4))[0][12]
print(f'Region {reg} with SNR {snr3727[reg]}')

inspectrum = sitspec[reg]
inspectrum[np.isnan(inspectrum)] = 0

wave3729 = 3728.815
c = 299792
red3729 = 1/((wave3729*(data[reg]['HA6562_VEL']+galvel)/(c) + wave3729) * 10**-8)

n1 = [np.where(sitlam > red3729 - 450)[0][0],np.where(sitlam > red3729 - 175)[0][0]]
n2 = [np.where(sitlam > red3729 + 175)[0][0], np.where(sitlam > red3729 + 700)[0][0]]

fit_lam = np.concatenate([sitlam[n2[0]:n2[1]], sitlam[n1[0]:n1[1]]])
fit_spec = np.concatenate([inspectrum[n2[0]:n2[1]], inspectrum[n1[0]:n1[1]]])
s_fit = np.polyfit(fit_lam, fit_spec, 1)
sfit_func = np.poly1d(s_fit)

noisestd = (np.std(inspectrum[n1[0]:n1[1]]) + np.std(inspectrum[n2[0]:n2[1]])) / 2
inspectrum = inspectrum - sfit_func(sitlam)

flux = np.array(data['OII3727_FLUX_CORR'])[reg]

ax3.plot(1/(10**-8*sitlam), inspectrum, c='k')#, label='spectrum')
ax3.plot(1/(10**-8*sitlam), fit3727[reg], c='blue', label=f'{galaxy} Region {reg}')

#ax3.set_title(f'Region {reg} | SNR: {round(snr3727[reg], 2)}')
ax3.set_xlim(x_min, x_max)
max_spec = np.nanmax(sitspec[reg][(sitlam > 1/(3850*10**-8)) & (sitlam < 1/(3650*10**-8))])
min_spec = np.nanmin(sitspec[reg][(sitlam > 1/(3850*10**-8)) & (sitlam < 1/(3650*10**-8))])
ax3.set_ylim(min_spec*flux_bottom * 1.5, max_spec*flux_top)
ax3.legend(fontsize=12, loc='upper right')

ax3.fill_between(1/(10**-8*sitlam)[np.where(sitlam > red3729 - 450)[0][0]:np.where(sitlam > red3729 - 175)[0][0]],-10**6, 10**10, color='purple', alpha = 0.15)
ax3.fill_between(1/(10**-8*sitlam)[np.where(sitlam > red3729 + 175)[0][0]:np.where(sitlam > red3729 + 700)[0][0]],-10**6, 10**10, color='purple', alpha = 0.15)

plt.ticklabel_format(axis='y', style='sci', scilimits=(0,0))
fig.text(0.5, -0.05, r'Wavelength [$\AA$]', ha='center', fontsize=20)
fig.text(-0.03, 0.5, r'Flux [$\frac{erg}{cm^{2} \cdot s \cdot \AA}$]', va='center', rotation='vertical', fontsize=20)
plt.tight_layout()
#plt.legend()
plt.show()

fig.savefig(path_out+'OII3727_fit.png',dpi=300,bbox_inches='tight')

### Load SITELLE+MUSE H II catalog

In [ ]:
hiicat_sitelle = table.Table.read('/home/habjan/SITELLE/data/data_products/strong_line_MUSE+SITELLE.fits')

hiicat_v4 = table.Table.read('/home/habjan/SITELLE/sandbox_notebooks/HABJAN_Sitelle_papers/nebular_catalog_v4_kk.fits')


### Make catalog for plotting

In [ ]:
sn_cut = 5
#hii_oii = (hiicat_sitelle['OII3727_FLUX']/hiicat_sitelle['OII3727_FLUX_ERR'] > sn_cut) & (hiicat_sitelle['HII_class_v3']==1)
hii_oii = (hiicat_sitelle['OII3727_FLUX']/hiicat_sitelle['OII3727_FLUX_ERR'] > sn_cut) & (hiicat_sitelle['HII_class_v3']==1) & (hiicat_sitelle['SIII9068_FLUX']/hiicat_sitelle['SIII9068_FLUX_ERR'] > sn_cut)

hiicat = hiicat_sitelle[hii_oii]


# join the v4 and eric's catalog
hiicat['_orig_idx'] = np.arange(len(hiicat))
matched = join(
    hiicat,
    hiicat_v4,
    keys=('gal_name', 'region_ID'),
    join_type='left'
)
matched.sort('_orig_idx')
matched.remove_column('_orig_idx')


n2o2 = hiicat['NII6583_FLUX_CORR']/hiicat['OII3727_FLUX_CORR']
r=np.log10(n2o2)
met_kd02_n2o2 = np.log10(1.54020 +1.26602*r  + 0.167977*r**2 )+ 8.93

print('Number of HII regions: ', len(hiicat))



# join the v4 and eric's catalog
matched2 = join(
    hiicat_sitelle,
    hiicat_v4,
    keys=('gal_name', 'region_ID'),
    join_type='left'
)
print(np.sum((matched2['FLAG_NII5754']==1) & (matched2['NII5754_FLUX_2']/matched2['NII5754_FLUX_ERR_RMS']>3) ))

### Spearman rank coefficient and pearson coefficient

In [ ]:
o3o2=hiicat['OIII5006_FLUX_CORR']/hiicat['OII3727_FLUX_CORR']
o3o2_err=(hiicat['OIII5006_FLUX_CORR']+hiicat['OIII5006_FLUX_CORR_ERR'])/(hiicat['OII3727_FLUX_CORR']-hiicat['OII3727_FLUX_CORR_ERR'])-o3o2
s3s2=hiicat['SIII9068_FLUX_CORR']*2.5/(hiicat['SII6716_FLUX_CORR']+hiicat['SII6730_FLUX_CORR'])
s3s2_err=(hiicat['SIII9068_FLUX_CORR']+hiicat['SIII9068_FLUX_CORR_ERR'])*2.5/(hiicat['SII6716_FLUX_CORR']+hiicat['SII6730_FLUX_CORR']-hiicat['SII6716_FLUX_CORR_ERR']-hiicat['SII6730_FLUX_CORR_ERR'])-s3s2
r23 = (hiicat['OIII5006_FLUX_CORR']+hiicat['OII3727_FLUX_CORR'])/hiicat['HB4861_FLUX_CORR']
r23_err = (hiicat['OIII5006_FLUX_CORR']+hiicat['OII3727_FLUX_CORR']+hiicat['OIII5006_FLUX_CORR_ERR']+hiicat['OII3727_FLUX_CORR_ERR'])/(hiicat['HB4861_FLUX_CORR']-hiicat['HB4861_FLUX_CORR_ERR'])-r23

n2o2 = hiicat['NII6583_FLUX_CORR']/hiicat['OII3727_FLUX_CORR']

r=np.log10(n2o2)
met_kd02_n2o2 = np.log10(1.54020 +1.26602*r  + 0.167977*r**2 )+ 8.93


r, pval = pearsonr(o3o2, s3s2)

print(f"o3o2 vs s3s2, r = {r:.3f}, p = {pval:.3e}")


r, pval = pearsonr(o3o2, r23)
print(f"o3o2 vs r23, r = {r:.3f}, p = {pval:.3e}")

r, pval = pearsonr(s3s2, r23)
print(f"s3s2 vs r23, r = {r:.3f}, p = {pval:.3e}")


r, pval = pearsonr(o3o2, n2o2)
print(f"o3o2 vs n2o2, r = {r:.3f}, p = {pval:.3e}")
r, pval = pearsonr(s3s2, n2o2)
print(f"s3s2 vs n2o2, r = {r:.3f}, p = {pval:.3e}")

In [ ]:
plt.rcParams['font.family']='serif'
plt.rcParams['font.size']=14
plt.rcParams['lines.linewidth']='5'
plt.style.use('classic')

# Figure 3: [O III]/[O II] versus [S III]/[S II] with photoionization models

In [ ]:
gals = np.unique(hiicat_sitelle['gal_name'])
colors=['blue','green','red','cyan','purple']

fig, axs = plt.subplots(figsize=(8,6))

for i,g in enumerate(gals):

    hii_oiiG = (hiicat_sitelle['OII3727_FLUX']/hiicat_sitelle['OII3727_FLUX_ERR'] > sn_cut) & (hiicat_sitelle['HII_class_v3']==1) & \
     (hiicat_sitelle['SIII9068_FLUX']/hiicat_sitelle['SIII9068_FLUX_ERR'] > sn_cut) & (hiicat_sitelle['gal_name']==g)
    hiicatG = hiicat_sitelle[hii_oiiG]

    o3o2G=hiicatG['OIII5006_FLUX_CORR']/hiicatG['OII3727_FLUX_CORR']
    o3o2G_err=(hiicatG['OIII5006_FLUX_CORR']+hiicatG['OIII5006_FLUX_CORR_ERR'])/(hiicatG['OII3727_FLUX_CORR']-hiicatG['OII3727_FLUX_CORR_ERR'])-o3o2G

    s3s2G=hiicatG['SIII9068_FLUX_CORR']*2.5/(hiicatG['SII6716_FLUX_CORR']+hiicatG['SII6730_FLUX_CORR'])
    s3s2G_err=(hiicatG['SIII9068_FLUX_CORR']+hiicatG['SIII9068_FLUX_CORR_ERR'])*2.5/(hiicatG['SII6716_FLUX_CORR']+hiicatG['SII6730_FLUX_CORR']-hiicatG['SII6716_FLUX_CORR_ERR']-hiicatG['SII6730_FLUX_CORR_ERR'])-s3s2G

    axs.scatter(o3o2G,s3s2G,label=g,color=colors[i],marker='o',alpha=0.6)
    axs.errorbar(o3o2G,s3s2G,xerr=o3o2G_err,yerr=s3s2G_err,ls = 'None', \
            markeredgewidth=1, linewidth = 1,elinewidth=1, zorder = 0,\
             color='gray', marker='*',markeredgecolor = 'None',markersize = 0,alpha = 0.9)

#legend1 = plt.legend(handles=lines1,loc='lower right')
#plt.gca().add_artist(legend1)
legend1 = axs.legend(scatterpoints=1, loc='lower right')

#axs.legend(scatterpoints=1,loc='lower right')
axs.set_xlabel('[OIII]/[OII]',fontsize=16)
axs.set_ylabel('[SIII]/[SII]',fontsize=16)
axs.set_xscale('log')
axs.set_yscale('log')

axs.set_xlim(.02,3)
axs.set_ylim(.02,3)

# overplot the Kewley+2019 lines
lines=[]
norm = mcolors.Normalize(vmin=7.0, vmax=9.0)
cmap = cm.get_cmap('Reds')
for yy in np.arange(8.0,9.2,.2):
  xx= np.arange(-2,1,.01) #log10(OIII/OII)
  coeffs = [13.768, 9.494, -4.3223, -2.3431, -0.5769, 0.2794, 0.1574, 0.0890, 0.0311, 0.]
  logU_oiiioii = coeffs[0]+coeffs[1]*xx+coeffs[2]*yy+coeffs[3]*xx*yy+coeffs[4]*xx**2+coeffs[5]*yy**2+coeffs[6]*xx*yy**2+coeffs[7]*xx**2*yy+coeffs[8]*xx**3+coeffs[9]*yy**3
  r_oiiioii = xx

  xx= np.arange(-.5,.5,.01) # log10(SIII/SII)
  coeffs = [90.017, 21.934, -34.095, -5.0818, -1.4762, 4.1343, 0.3096, 0.1786, 0.1959, -0.1668]
  logU_siiisii = coeffs[0]+coeffs[1]*xx+coeffs[2]*yy+coeffs[3]*xx*yy+coeffs[4]*xx**2+coeffs[5]*yy**2+coeffs[6]*xx*yy**2+coeffs[7]*xx**2*yy+coeffs[8]*xx**3+coeffs[9]*yy**3
  r_siiisii = xx

  r_siiisii_matched = np.interp(logU_oiiioii, logU_siiisii, r_siiisii)

  color = cmap(norm(yy))

  line_new, = axs.plot(10**r_oiiioii, 10**r_siiisii_matched,label=f'{yy:.1f}',color=color)
  lines.append(line_new)
  axs.set_xscale('log')
  axs.set_yscale('log')
#ax.legend()
legend2 = plt.legend(handles=lines, loc="lower left",title='12+log(O/H)')
#plt.gca().add_artist(legend2)
axs.add_artist(legend1)
fig.savefig(path_out+'Figure_ionization_models.png',dpi=300,bbox_inches='tight')


### Helper function: plotty for metallicity gradients

In [ ]:
def fit_metalgrad(x,y):
    coef = np.polyfit(x,y,1)
    return coef[0],coef[1]

def plotty(y,axx,show_all=False,show_title=False,useoiiioii=False):
    gals = ['NGC0628','NGC2835','NGC3627']

    for i,g in enumerate(gals):
        hii_oiiG = (hiicat_sitelle['OII3727_FLUX']/hiicat_sitelle['OII3727_FLUX_ERR'] > sn_cut) & (hiicat_sitelle['HII_class_v3']==1) & \
        (hiicat_sitelle['SIII9068_FLUX']/hiicat_sitelle['SIII9068_FLUX_ERR'] > sn_cut) & (hiicat_sitelle['gal_name']==g)
        hiicatG = hiicat_sitelle[hii_oiiG]

        hii_oiiG_all = (hiicat_sitelle['HII_class_v3']==1) & (hiicat_sitelle['met_scal']>8.0) & \
        (hiicat_sitelle['SIII9068_FLUX']/hiicat_sitelle['SIII9068_FLUX_ERR'] > sn_cut) & (hiicat_sitelle['gal_name']==g)
        hiicatG_all = hiicat_sitelle[hii_oiiG_all]

        s3s2 = hiicatG['SIII9068_FLUX_CORR']*2.5/(hiicatG['SII6716_FLUX_CORR']+hiicatG['SII6730_FLUX_CORR'])
        if useoiiioii:
            s3s2= hiicatG['OIII5006_FLUX_CORR']*2.5/(hiicatG['OII3727_FLUX_CORR'])

        sc=axx[i].scatter(hiicatG['r_reff'],y[hii_oiiG],label=g,c=s3s2,marker='o',vmin=0,vmax=1,cmap='plasma',edgecolors='none')
        if show_all:
            axx[i].scatter(hiicatG_all['r_reff'],y[hii_oiiG_all],color='grey',marker='o',alpha=0.6,edgecolors='none',zorder=0)

        coef1,coef2 = fit_metalgrad(hiicatG['r_reff'],y[hii_oiiG])
        poly1d_fn = np.poly1d([coef1,coef2])
        xx=np.arange(0,2.5,.1)
        axx[i].plot(xx, poly1d_fn(xx), 'k',linewidth=2,zorder=5,alpha=0.5)

        delt=y[hii_oiiG] - poly1d_fn(hiicatG['r_reff'])
        axx[i].text(
          0.98, 0.98, f"$\sigma$(O/H)={np.std(delt):.3f}",
          transform=axx[i].transAxes,
          ha='right',
          va='top'
        )

        ymin, ymax = axx[i].get_ylim()
        yav=(ymin+ymax)/2.
        axx[i].set_ylim(yav-.4,yav+.4)
        axx[i].set_xlim(0,2.5)

        if show_title:
            axx[i].set_title(g)

    return sc

# Figure 4: Metallicity gradients for general literature calibrations

### Strong line metallicity calculation

In [ ]:
n2o2 = hiicat_sitelle['NII6583_FLUX_CORR']/hiicat_sitelle['OII3727_FLUX_CORR']
r=np.log10(n2o2)
met_kd02_n2o2 = np.log10(1.54020 +1.26602*r  + 0.167977*r**2 )+ 8.93

n2o2 = (hiicat_sitelle['NII6583_FLUX_CORR']+hiicat_sitelle['NII6583_FLUX_CORR_ERR'])/(hiicat_sitelle['OII3727_FLUX_CORR']-hiicat_sitelle['OII3727_FLUX_CORR_ERR'])
r=np.log10(n2o2)
met_kd02_n2o2_err = (np.log10(1.54020 +1.26602*r  + 0.167977*r**2 )+ 8.93)-met_kd02_n2o2

### Make plot

In [ ]:
fig, axs = plt.subplots(6,3,figsize=(12,14),sharex=True,gridspec_kw={'hspace': 0, 'wspace': 0.2})
axsF = axs.flatten()

plotty(met_kd02_n2o2,axsF[0:3],show_title=True)
axsF[0].set_ylabel('KD02 [N2O2]')
ymin, ymax = axsF[0].get_ylim()
axsF[0].errorbar([2.],[(ymax-ymin)*0.8+ymin],yerr=[.03],marker='.',color='black')

plotty(hiicat_sitelle['met_kk04'],axsF[3:6])
axsF[3].set_ylabel('KK04 [R23]')
ymin, ymax = axsF[3].get_ylim()
axsF[3].errorbar([2.],[(ymax-ymin)*0.8+ymin],yerr=[np.median(hiicat_sitelle['met_kk04_err'][~hiicat_sitelle['met_kk04_err'].mask])],marker='.',color='black')

plotty(hiicat_sitelle['met_p05'],axsF[6:9])
axsF[6].set_ylabel('PT05 [R23]')
ymin, ymax = axsF[6].get_ylim()
axsF[6].errorbar([2.],[(ymax-ymin)*0.8+ymin],yerr=[np.median(hiicat_sitelle['met_p05_err'][~hiicat_sitelle['met_p05_err'].mask])],marker='.',color='black')

plotty(hiicat_sitelle['met_rcal'],axsF[9:12],show_all=True)
axsF[9].set_ylabel('PG16-Rcal')
ymin, ymax = axsF[9].get_ylim()
axsF[9].errorbar([2.],[(ymax-ymin)*0.8+ymin],yerr=[np.median(hiicat_sitelle['met_rcal_err'][~hiicat_sitelle['met_rcal_err'].mask])],marker='.',color='black')

plotty(hiicat_sitelle['met_scal'],axsF[12:15],show_all=True)
axsF[12].set_ylabel('PG16-Scal')
ymin, ymax = axsF[12].get_ylim()
axsF[12].errorbar([2.],[(ymax-ymin)*0.8+ymin],yerr=[np.median(hiicat_sitelle['met_scal_err'][~hiicat_sitelle['met_scal_err'].mask])],marker='.',color='black')

sc=plotty(hiicat_sitelle['met_d16'],axsF[15:18],show_all=True)
axsF[15].set_ylabel('D16 [N2S2]')
ymin, ymax = axsF[15].get_ylim()
axsF[15].errorbar([2.],[(ymax-ymin)*0.8+ymin],yerr=[np.median(hiicat_sitelle['met_d16_err'][~hiicat_sitelle['met_d16_err'].mask])],marker='.',color='black')

for ax in axsF:
    ymin, ymax = ax.get_ylim()
    yrange = ymax - ymin
    ax.set_ylim(ymin - 0.12 * yrange, ymax + 0.12 * yrange)

cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])  # [left, bottom, width, height]
cbar = fig.colorbar(sc, cax=cax)
#cbar = fig.colorbar(sc, ax=axsF[-1])
cbar.set_label('[SIII]/[SII]')   # or whatever met_d16 represents

fig.supxlabel(r'radius [r$_{eff}$]', fontsize=16, y=0.04)
fig.supylabel(r'12+log(O/H)',fontsize=16)

fig.savefig(path_out+'Figure_gradient1.png', dpi=300,bbox_inches='tight')


# Figure 5: Metallicity gradients for DESIRED (R26) calibrations

# Figure 6: N/O as a function of 12+log(O/H) [PT05 R23]

### SNR cuts and strong line computation

In [ ]:
sn_cut = 5

n2o2 = hiicat_sitelle['NII6583_FLUX_CORR']/hiicat_sitelle['OII3727_FLUX_CORR']
r=np.log10(n2o2)

n2 = hiicat_sitelle['NII6583_FLUX_CORR']*1.33/hiicat_sitelle['OII3727_FLUX_CORR']
r2 = hiicat_sitelle['OII3727_FLUX_CORR']/hiicat_sitelle['HB4861_FLUX_CORR']

met_kd02_n2o2 = np.log10(1.54020 +1.26602*r  + 0.167977*r**2 )+ 8.93

no_pc09 = 0.93*r-0.20
no_pg16 = -0.657 - 0.201 * np.log10(n2)  + (0.742 - 0.075* np.log10(n2)) * np.log10(n2/r2)

no_pg16_err=(hiicat_sitelle['NII6583_FLUX_CORR'])*0.

for i, hh in enumerate(hiicat_sitelle):
    if (hh['OII3727_FLUX']/hh['OII3727_FLUX_ERR'] > sn_cut):
        tmp = np.zeros(100)
        for n in range(100):
            n2 = (hh['NII6583_FLUX_CORR']+np.random.normal()*hh['NII6583_FLUX_CORR_ERR'])*1.33/(hh['OII3727_FLUX_CORR']+np.random.normal()*hh['OII3727_FLUX_CORR_ERR'])
            r2 = (hh['OII3727_FLUX_CORR']+np.random.normal()*hh['OII3727_FLUX_CORR_ERR'])/(hh['HB4861_FLUX_CORR']+np.random.normal()*hh['HB4861_FLUX_CORR_ERR'])
            no_pg16R = -0.657 - 0.201 * np.log10(n2)  + (0.742 - 0.075* np.log10(n2)) * np.log10(n2/r2)
            tmp[n]=no_pg16R

        no_pg16_err[i] = np.std(tmp)


hiicat = hiicat_sitelle[hii_oii]


### Make plot

In [ ]:
gals = np.unique(hiicat_sitelle['gal_name'])
colors=['blue','green','red','cyan','purple']

fig, axs = plt.subplots(figsize=(8,6))

for i,g in enumerate(gals):
    hii_oiiG = (hiicat_sitelle['OII3727_FLUX']/hiicat_sitelle['OII3727_FLUX_ERR'] > sn_cut) & (hiicat_sitelle['HII_class_v3']==1) & \
     (hiicat_sitelle['SIII9068_FLUX']/hiicat_sitelle['SIII9068_FLUX_ERR'] > sn_cut) & (hiicat_sitelle['gal_name']==g)
    hiicatG = hiicat_sitelle[hii_oiiG]

    axs.scatter(hiicat_sitelle['met_p05'][hii_oiiG],no_pg16[hii_oiiG],marker='o',color=colors[i],label=g)

    axs.errorbar(hiicat_sitelle['met_p05'][hii_oiiG],no_pg16[hii_oiiG],xerr=hiicat_sitelle['met_p05_err'][hii_oiiG],yerr=no_pg16_err[hii_oiiG],ls = 'None', \
        markeredgewidth=1, linewidth = 1,elinewidth=1, zorder = 0,\
          color='gray', marker='*',markeredgecolor = 'None',markersize = 0,alpha = 0.9)


axs.legend(loc='upper left',fontsize=16,scatterpoints=1)
axs.set_xlim(7.8,8.9)
axs.set_ylim(-1.7,0.2)
fig.supxlabel('12+log(O/H) [PT05-R23]',fontsize=16)
fig.supylabel('N/O',fontsize=16)
fig.savefig(path_out+'Figure_NO-OH.png', dpi=300,bbox_inches='tight')

In [ ]:
sn_cut = 5

n2o2 = hiicat_sitelle['NII6583_FLUX_CORR']/hiicat_sitelle['OII3727_FLUX_CORR']
r=np.log10(n2o2)

no_f22 = -0.102*r**2 + 0.528*r - 0.634
no_f22_err = np.zeros(len(no_f22))

for i, hh in enumerate(hiicat_sitelle):
    if (hh['OII3727_FLUX']/hh['OII3727_FLUX_ERR'] > sn_cut):
        tmp = np.zeros(100)
        for n in range(100):
            n2o2_i = np.log10(np.random.normal(hiicat_sitelle['NII6583_FLUX_CORR'][i], hiicat_sitelle['NII6583_FLUX_CORR_ERR'][i]) / np.random.normal(hiicat_sitelle['OII3727_FLUX_CORR'][i], hiicat_sitelle['OII3727_FLUX_CORR_ERR'][i]))
            no_f22_i = np.random.normal(-0.102, 0.018)*n2o2_i**2 + np.random.normal(0.528, 0.019)*n2o2_i - np.random.normal(0.634, 0.006)
            tmp[n]=no_f22_i

        no_f22_err[i] = np.std(tmp)

In [ ]:
gals = np.unique(hiicat_sitelle['gal_name'])
colors=['blue','green','red','cyan','purple']

fig, axs = plt.subplots(figsize=(8,6))

for i,g in enumerate(gals):
    hii_oiiG = (hiicat_sitelle['OII3727_FLUX']/hiicat_sitelle['OII3727_FLUX_ERR'] > sn_cut) & (hiicat_sitelle['HII_class_v3']==1) & \
     (hiicat_sitelle['SIII9068_FLUX']/hiicat_sitelle['SIII9068_FLUX_ERR'] > sn_cut) & (hiicat_sitelle['gal_name']==g)
    hiicatG = hiicat_sitelle[hii_oiiG]

    axs.scatter(hiicat_sitelle['met_p05'][hii_oiiG],no_f22[hii_oiiG],marker='o',color=colors[i],label=g)

    axs.errorbar(hiicat_sitelle['met_p05'][hii_oiiG],no_f22[hii_oiiG],xerr=hiicat_sitelle['met_p05_err'][hii_oiiG],yerr=no_f22_err[hii_oiiG],ls = 'None', \
        markeredgewidth=1, linewidth = 1,elinewidth=1, zorder = 0,\
          color='gray', marker='*',markeredgecolor = 'None',markersize = 0,alpha = 0.9)


axs.legend(loc='upper left',fontsize=16,scatterpoints=1)
axs.set_xlim(7.8,8.9)
axs.set_ylim(-1.7,0.2)
fig.supxlabel('12+log(O/H) [PT05-R23]',fontsize=16)
fig.supylabel('N/O [F22-N2O2]',fontsize=16)
fig.savefig(path_out+'Figure_NO-OH_F22.png', dpi=300,bbox_inches='tight')

### Helper functions: plot_pts and plot_one for strong-line comparison

In [ ]:
def plot_pts(x0,y0,axs,c_max=100):
    # define fixed binning
    binx=np.arange(7.0,10.0,.05)
    biny=np.arange(7.0,10.0,.05)

# define colorbar ranges
# counts
    c_cmap='viridis'
    c_min=1
#    c_max=100

#set requirement on minimum number in a bin
    minbin_count=1


    ii=(np.isfinite(x0)) & (np.isfinite(y0))

    x = x0[ii]
    y = y0[ii]

    # count
    Hc, xedges, yedges, binnumber = stats.binned_statistic_2d(x, y, x, 'count', bins=[binx, biny])
    H = np.ma.masked_where(Hc<=minbin_count, Hc) #masking where there was no data
    XX, YY = np.meshgrid(xedges, yedges)
    #c = axs.pcolormesh(XX,YY,np.log10(H.T),cmap=c_cmap,vmin=np.log10(c_min),vmax=np.log10(c_max))
    c=None
    axs.scatter(x,y,marker='.',color='k')

    xmid = np.nanmedian(x)
    ymid = np.ma.median(y)


    axs.set_xlim(xmid-0.5,xmid+0.5)
    axs.set_ylim(ymid-0.5,ymid+0.5)

    axs.plot([0,10],[0,10],color='k',linestyle='dashed')

    mask = (~x.mask) & (~y.mask)

    rho, p_spear = spearmanr(x[mask], y[mask])
    r, p_pear = pearsonr(x[mask], y[mask])

#    axs.text(.1,.9,f"ρ = {rho:.2f} (p={p_spear:.2e})",transform=axs.transAxes)
#    axs.text(.1,.8,f"r = {r:.2f} (p={p_pear:.2e})",transform=axs.transAxes)

    axs.text(.1,.9,f"ρ = {rho:.2f}",transform=axs.transAxes)
    axs.text(.1,.8,f"r = {r:.2f}",transform=axs.transAxes)

    return c


def plot_one(x0,y0,axs,c_max=100):
    # define fixed binning
    binx=np.arange(7.0,10.0,.05)
    biny=np.arange(7.0,10.0,.05)

# define colorbar ranges
# counts
    c_cmap='viridis'
    c_min=1
#    c_max=100

#set requirement on minimum number in a bin
    minbin_count=1


    ii=(np.isfinite(x0)) & (np.isfinite(y0))

    x = x0[ii]
    y = y0[ii]

    # count
    Hc, xedges, yedges, binnumber = stats.binned_statistic_2d(x, y, x, 'count', bins=[binx, biny])
    H = np.ma.masked_where(Hc<=minbin_count, Hc) #masking where there was no data
    XX, YY = np.meshgrid(xedges, yedges)
    c = axs.pcolormesh(XX,YY,np.log10(H.T),cmap=c_cmap,vmin=np.log10(c_min),vmax=np.log10(c_max))

    xmid = np.median(x)
    ymid = np.median(y)

    axs.set_xlim(xmid-0.5,xmid+0.5)
    axs.set_ylim(ymid-0.5,ymid+0.5)

    axs.plot([0,10],[0,10],color='k',linestyle='dashed')

    return c


### Variable definitions for strong-line comparison

In [ ]:
v1 = matched['met_scal_2']
v1_label = 'PG16-Scal'
v2 = matched['met_rcal']
v2_label = 'PG16-Rcal'
#v2 = matched['met_rcal']
#v2_label = 'PG16-Rcal'
v3 = matched['met_kk04']
v3_label = 'KK04 [R23]'
v4 = matched['met_p05']
v4_label = 'PT05 [R23]'
v5 = met_kd02_n2o2[np.where(hii_oii)[0]]
v5_label = 'KD02 [N2O2]'
#v5 = matched['met_m91']
#v5_label = 'M91 [R23,O32]'
v6 = matched['met_d02']
v6_label = 'D02 [N2]'
#v7 = matched['met_rs2Dcal']
#v7_label = 'rs2Dcal'
v7 = matched['met_d16']
v7_label = 'D16 [N2S2]'
v8 = matched['met_m13_o3n2']
v8_label = 'M13 [O3N2]'

# Figure 7: Comparison of MD23 [Te,NII] with strong line calibrations

In [ ]:
coeff1 = -1.19
coeff2 = 9.68
met_jemd = coeff1*1e-4 * matched['TEM_NII'].data + coeff2

high_conf = (matched['FLAG_NII5754']==1) & (matched['NII5754_FLUX_2']/matched['NII5754_FLUX_ERR_RMS']>3) & (v1>0)
print(np.sum(high_conf))
#yy=hiicat['met_md23'] #from Eric
yy=met_jemd #from Matilde

fig, axs = plt.subplots(1,6,figsize=(16,3),gridspec_kw={'hspace': 0, 'wspace': 0})

c = plot_pts(v5[high_conf],yy[high_conf], axs[0],c_max=10)
axs[0].set_xlabel(v5_label,size=16,rotation=0,labelpad=0)
plot_pts(v3[high_conf], yy[high_conf],axs[1],c_max=10)
axs[1].set_xlabel(v3_label,size=16,rotation=0,labelpad=0)
axs[1].set_yticklabels([])
plot_pts(v4[high_conf], yy[high_conf],axs[2],c_max=10)
axs[2].set_xlabel(v4_label,size=16,rotation=0,labelpad=0)
axs[2].set_yticklabels([])
plot_pts(v1[high_conf], yy[high_conf],axs[3],c_max=10)
axs[3].set_xlabel(v1_label,size=16,rotation=0,labelpad=0)
axs[3].set_yticklabels([])
plot_pts(v2[high_conf], yy[high_conf],axs[4],c_max=10)
axs[4].set_xlabel(v2_label,size=16,rotation=0,labelpad=0)
axs[4].set_yticklabels([])
plot_pts(v7[high_conf], yy[high_conf],axs[5],c_max=10)
axs[5].set_xlabel(v7_label,size=16,rotation=0,labelpad=0)
axs[5].set_yticklabels([])
#plot_pts(v7[high_conf], yy[high_conf],axs[6],c_max=10)
#axs[6].set_xlabel(v7_label,size=16,rotation=0,labelpad=0)
#axs[6].set_yticklabels([])

axs[0].set_ylabel(r'MD23 [Te$_{NII}$]',size=16)

fig.supxlabel('12+log(O/H)',fontsize=16,y=-0.2)
fig.supylabel('12+log(O/H)',fontsize=16,x=0.05)
fig.savefig(path_out+'Figure_vs_md23.png', dpi=300,bbox_inches='tight')

# Figure 8: [O III]/[O II] and [S III]/[S II] versus R23 and 12+log(O/H)

### MD23 arrays

In [ ]:
met_md23 = np.array(hiicat['met_md23'])
met_md23_err = np.array(hiicat['met_md23_err'])

### Make plot

In [ ]:
fig, axs = plt.subplots(2,2,figsize=(13,10),sharey=True,gridspec_kw={'hspace':0, 'wspace':0})
fig.set_facecolor('white')

sc = axs[0, 0].scatter(r23,o3o2,c=hiicat['EBV'],marker='o',s=50,alpha=1,cmap='copper_r',vmin=0,vmax=1,edgecolors='none')
sc = axs[1, 0].scatter(r23,s3s2,c=hiicat['EBV'],marker='o',s=50,alpha=1,cmap='copper_r',vmin=0,vmax=1,edgecolors='none')


axs[0,0].errorbar(r23,o3o2,xerr=r23_err,yerr=o3o2_err,ls = 'None', \
        markeredgewidth=1, linewidth = 1,elinewidth=1, zorder = 0,\
          color='gray', marker='*',markeredgecolor = 'None',markersize = 0,alpha = 0.9)
axs[1,0].errorbar(r23,s3s2,xerr=r23_err,yerr=s3s2_err,ls = 'None', \
        markeredgewidth=1, linewidth = 1,elinewidth=1, zorder = 0,\
          color='gray', marker='*',markeredgecolor = 'None',markersize = 0,alpha = 0.9)

sc = axs[0, 1].scatter(met_md23,o3o2,c=hiicat['EBV'],marker='o',s=50,alpha=1,cmap='copper_r',vmin=0,vmax=1,edgecolors='none')
sc = axs[1, 1].scatter(met_md23,s3s2,c=hiicat['EBV'],marker='o',s=50,alpha=1,cmap='copper_r',vmin=0,vmax=1,edgecolors='none')

axs[0, 1].errorbar(met_md23,o3o2,xerr=met_md23_err,yerr=o3o2_err,ls = 'None', \
        markeredgewidth=1, linewidth = 1,elinewidth=1, zorder = 0,\
          color='gray', marker='*',markeredgecolor = 'None',markersize = 0,alpha = 0.9)
axs[1, 1].errorbar(met_md23,s3s2,xerr=met_md23_err,yerr=s3s2_err,ls = 'None', \
        markeredgewidth=1, linewidth = 1,elinewidth=1, zorder = 0,\
          color='gray', marker='*',markeredgecolor = 'None',markersize = 0,alpha = 0.9)

cax = fig.add_axes([0.15, 0.93, 0.7, 0.02])  # [left, bottom, width, height]
cbar = fig.colorbar(sc, cax=cax, orientation='horizontal')
cbar.set_label('E(B-V) [mag]',fontsize=16)   # or whatever met_d16 represents
cbar.ax.xaxis.set_ticks_position('top')   # ticks on top
cbar.ax.xaxis.set_label_position('top')   # label on top

axs[0, 0].set_ylabel('[OIII]/[OII]',fontsize=16)
axs[1, 0].set_ylabel('[SIII]/[SII]',fontsize=16)
axs[1, 0].set_xlabel('R23',fontsize=16)
axs[1, 1].set_xlabel('12 + log(O/H)',fontsize=16)

axs[0, 0].set_xscale('log')
axs[0, 0].set_yscale('log')
axs[1, 0].set_xscale('log')
axs[1, 0].set_yscale('log')

axs[0, 1].set_yscale('log')
axs[1, 1].set_yscale('log')

axs[0, 0].set_ylim(.03,3)
axs[1, 0].set_ylim(.03,3)
axs[0, 1].set_ylim(.03,3)
axs[1, 1].set_ylim(.03,3)

axs[0, 0].set_xlim(.3,7)
axs[1, 0].set_xlim(.3,7)
axs[0, 1].set_xlim(8.35,9.05)
axs[1, 1].set_xlim(8.35,9.05)


# overplot the Kewley+2019 lines
lines=[]
norm = mcolors.Normalize(vmin=-5, vmax=-2)
cmap = cm.get_cmap('YlGnBu')#Blues')
for uu in np.arange(-4,-2,.5):
  xxs_o=[]
  yys_o=[]
  xxs_s=[]
  yys_s=[]
  for yy in np.arange(8.0,9.1,.1):
    xx = np.arange(-2,1,0.01)
    coeffs = [13.768, 9.494, -4.3223, -2.3431, -0.5769, 0.2794, 0.1574, 0.0890, 0.0311, 0.]
    logU_oiiioii = coeffs[0]+coeffs[1]*xx+coeffs[2]*yy+coeffs[3]*xx*yy+coeffs[4]*xx**2+coeffs[5]*yy**2+coeffs[6]*xx*yy**2+coeffs[7]*xx**2*yy+coeffs[8]*xx**3+coeffs[9]*yy**3
    idx = np.argmin(np.abs(logU_oiiioii - (uu)))

    #color = cmap(norm(yy))
    #axs[0,1].scatter(yy,10**xx[idx],color=color) #r_oiiioii = xx
    #print(yy,10**xx[idx])
    xxs_o.append(yy)
    yys_o.append(10**xx[idx])

    xx= np.arange(-.5,.5,.01) # log10(SIII/SII)
    coeffs = [90.017, 21.934, -34.095, -5.0818, -1.4762, 4.1343, 0.3096, 0.1786, 0.1959, -0.1668]
    logU_siiisii = coeffs[0]+coeffs[1]*xx+coeffs[2]*yy+coeffs[3]*xx*yy+coeffs[4]*xx**2+coeffs[5]*yy**2+coeffs[6]*xx*yy**2+coeffs[7]*xx**2*yy+coeffs[8]*xx**3+coeffs[9]*yy**3
    idx = np.argmin(np.abs(logU_siiisii - (uu)))
    xxs_s.append(yy)
    yys_s.append(10**xx[idx])


  color=cmap(norm(uu))
  axs[0,1].plot(xxs_o,yys_o,color=color,linewidth=3)
  line_new, = axs[1,1].plot(xxs_s,yys_s,color=color,linewidth=3,label=f'{uu:.1f}')
  lines.append(line_new)

legend2 = axs[1,1].legend(handles=lines, loc="lower right",title='log U')

fig.savefig(path_out+'Figure_ionization7_models.png', dpi=300,bbox_inches='tight')

# Figure 9: [O II]/Hβ versus [S II]/Hα for unclassified objects

### Make booleans and collect stats

In [ ]:
sn_cut = 5
has_oii = (hiicat_sitelle['OII3727_FLUX']/hiicat_sitelle['OII3727_FLUX_ERR'] > sn_cut) & (hiicat_sitelle['flag_edge']==0) & (hiicat_sitelle['flag_star']==0)


nothii_oii = (hiicat_sitelle['OII3727_FLUX']/hiicat_sitelle['OII3727_FLUX_ERR'] > sn_cut) & (hiicat_sitelle['HII_class_v3']!=1) & (hiicat_sitelle['flag_edge']==0) & (hiicat_sitelle['flag_star']==0)
hii_oii = (hiicat_sitelle['OII3727_FLUX']/hiicat_sitelle['OII3727_FLUX_ERR'] > sn_cut) & (hiicat_sitelle['HII_class_v3']==1) & (hiicat_sitelle['flag_edge']==0) & (hiicat_sitelle['flag_star']==0)

cat = hiicat_sitelle[nothii_oii]
hiicat = hiicat_sitelle[hii_oii]

print('number of [OII] detections:', np.sum(has_oii), np.sum(has_oii)/np.sum(hiicat_sitelle['HII_class_v3']==1))
print('number of [OII] detections, hii regions:',np.sum(hii_oii))
print('number of [OII] detections, not HII regions:', len(cat))
print('fraction of [OII] detections:',np.sum(nothii_oii)/(np.sum(nothii_oii)+np.sum(hii_oii)))
print('number of nebulae in all 5 galaxies:',len(hiicat_sitelle))
print('number of HII regions in all 5 galaxies:', np.sum(hiicat_sitelle['HII_class_v3']==1))


### Plots data

In [ ]:
plt.scatter((hiicat['SII6716_FLUX']+hiicat['SII6730_FLUX'])/hiicat['HA6562_FLUX'],hiicat['OII3727_FLUX_CORR']/hiicat['HB4861_FLUX_CORR'],label='HII regions',color='k',marker='.')
plt.scatter((cat['SII6716_FLUX']+cat['SII6730_FLUX'])/cat['HA6562_FLUX'],cat['OII3727_FLUX_CORR']/cat['HB4861_FLUX_CORR'],label='Unclassified',color='red')
#plt.scatter((cat_matches['SII6716_FLUX']+cat_matches['SII6730_FLUX'])/cat_matches['HA6562_FLUX'],cat_matches['OII3727_FLUX_CORR']/cat_matches['HB4861_FLUX_CORR'],label='SNRs',color='red')

cc = hiicat
x= (cc['SII6716_FLUX']+cc['SII6730_FLUX'])/cc['HA6562_FLUX']
xerr= (cc['SII6716_FLUX']+cc['SII6730_FLUX']+cc['SII6716_FLUX_ERR']+cc['SII6730_FLUX_ERR'])/(cc['HA6562_FLUX']-cc['HA6562_FLUX_ERR'])-x
y=cc['OII3727_FLUX_CORR']/cc['HB4861_FLUX_CORR']
yerr=(cc['OII3727_FLUX_CORR']+cc['OII3727_FLUX_CORR_ERR'])/(cc['HB4861_FLUX_CORR']-cc['HB4861_FLUX_CORR_ERR'])-y

plt.errorbar(x,y,xerr=xerr,yerr=yerr,ls = 'None', \
        markeredgewidth=1, linewidth = 1,elinewidth=1, zorder = 0,\
          color='gray', marker='*',markeredgecolor = 'None',markersize = 0,alpha = 0.9)

cc = cat
x= (cc['SII6716_FLUX']+cc['SII6730_FLUX'])/cc['HA6562_FLUX']
xerr= (cc['SII6716_FLUX']+cc['SII6730_FLUX']+cc['SII6716_FLUX_ERR']+cc['SII6730_FLUX_ERR'])/(cc['HA6562_FLUX']-cc['HA6562_FLUX_ERR'])-x
y=cc['OII3727_FLUX_CORR']/cc['HB4861_FLUX_CORR']
yerr=(cc['OII3727_FLUX_CORR']+cc['OII3727_FLUX_CORR_ERR'])/(cc['HB4861_FLUX_CORR']-cc['HB4861_FLUX_CORR_ERR'])-y

plt.errorbar(x,y,xerr=xerr,yerr=yerr,ls = 'None', \
        markeredgewidth=1, linewidth = 1,elinewidth=1, zorder = 0,\
          color='gray', marker='*',markeredgecolor = 'None',markersize = 0,alpha = 0.9)


plt.xlabel(r'[SII]/H$\alpha$',fontsize=16)
plt.ylabel(r'[OII]/H$\beta$',fontsize=16)

plt.legend(scatterpoints=1,loc='lower right')
plt.xscale('log')
plt.yscale('log')
plt.xlim(.1,2)
plt.ylim(.15,20)

plt.savefig(path_out+'Figure_SNRs.png', dpi=300,bbox_inches='tight')

# [O II] detection plots

In [ ]:
sn_cut = 5

good_region = ((hiicat_sitelle['flag_edge'] == 0) & (hiicat_sitelle['flag_star'] == 0))

has_oii = ((hiicat_sitelle['OII3727_FLUX'] / hiicat_sitelle['OII3727_FLUX_ERR'] > sn_cut) & good_region)

no_oii = ((hiicat_sitelle['OII3727_FLUX'] / hiicat_sitelle['OII3727_FLUX_ERR'] <= sn_cut) & good_region)

hii = hiicat_sitelle['HII_class_v3'] == 1

has_oii_hii = has_oii & hii
no_oii_hii = no_oii & hii

In [ ]:
cat = hiicat_sitelle

N2 = np.log10(cat['NII6583_FLUX_CORR'] / cat['HA6562_FLUX_CORR'])
O3 = cat['OIII5006_FLUX_CORR'] / cat['HB4861_FLUX_CORR']
N2_linear = cat['NII6583_FLUX_CORR'] / cat['HA6562_FLUX_CORR']

O3N2 = np.log10(O3 / N2_linear)

SII = cat['SII6716_FLUX_CORR'] + cat['SII6730_FLUX_CORR']
SIII = cat['SIII9068_FLUX_CORR'] #* 2.5

S3S2 = SIII / SII
logS3S2 = np.log10(S3S2)

valid_n2 = np.isfinite(N2)

valid_o3n2 = (
    np.isfinite(O3N2) &
    (cat['OIII5006_FLUX_CORR'] > 0) &
    (cat['HB4861_FLUX_CORR'] > 0) &
    (cat['NII6583_FLUX_CORR'] > 0) &
    (cat['HA6562_FLUX_CORR'] > 0)
)

valid_s3s2 = (
    np.isfinite(logS3S2) &
    (SIII > 0) &
    (SII > 0)
)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

# --------------------------------------------------
# Left: N2 vs [SIII]/[SII]
# --------------------------------------------------
ax = axs[0]

m1 = has_oii_hii & valid_n2 & valid_s3s2
m2 = no_oii_hii & valid_n2 & valid_s3s2

ax.scatter(N2[m2], logS3S2[m2],
           color='lightgray', edgecolor='k', linewidth=0.3, s=35,)

ax.scatter(N2[m1], logS3S2[m1],
           color='tab:blue', edgecolor='k', linewidth=0.3, s=35,)

#ax.set_xlabel(r'$\log([\mathrm{NII}]\lambda6584/\mathrm{H}\alpha)$', fontsize=14)
ax.set_xlabel(r'N2', fontsize=14)
ax.set_ylabel(r'$\log([\mathrm{SIII}]/[\mathrm{SII}])$', fontsize=14)

# --------------------------------------------------
# Right: N2 vs O3N2
# --------------------------------------------------
ax = axs[1]

m1 = has_oii_hii & valid_n2 & valid_o3n2
m2 = no_oii_hii & valid_n2 & valid_o3n2

ax.scatter(N2[m2], O3N2[m2], 
           color='lightgray', edgecolor='k', linewidth=0.3, s=35, label=r'No [OII] detection')

ax.scatter(N2[m1], O3N2[m1],
           color='tab:blue', edgecolor='k', linewidth=0.3, s=35, label=r'[OII] detected')

#ax.set_xlabel(r'$\log([\mathrm{NII}]\lambda6584/\mathrm{H}\alpha)$', fontsize=14)
ax.set_xlabel(r'N2', fontsize=14)
#ax.set_ylabel(r'$\log([\mathrm{OIII}]/\mathrm{H}\beta)/([\mathrm{NII}]/\mathrm{H}\alpha)$', fontsize=14)
ax.set_ylabel(r'O3 - N2', fontsize=14)

for ax in axs:
    ax.legend(frameon=False, scatterpoints=1, loc='lower left')
    ax.tick_params(axis='both', labelsize=12)

plt.tight_layout()
plt.savefig(path_out+'Figure_OII_detection.png', dpi=300,bbox_inches='tight')

# Figure 10: Comparison of SITELLE [O II] flux with SIGNALS and KCWI literature values

### Import data

In [ ]:
sig_eric_oii = np.load('/home/habjan/SITELLE/sandbox_notebooks/sittelle_analysis/dig_subtraction/catalog_comp_arrays/SIGNALS_comp_eric_NGC0628.npy', allow_pickle=True).astype(np.float64)
sig_laurie_oii = np.load('/home/habjan/SITELLE/sandbox_notebooks/sittelle_analysis/dig_subtraction/catalog_comp_arrays/SIGNALS_comp_laurie_NGC0628.npy', allow_pickle=True).astype(np.float64)
sig_ebv = np.load('/home/habjan/SITELLE/sandbox_notebooks/sittelle_analysis/dig_subtraction/catalog_comp_arrays/SIGNALS_comp_ebv_NGC0628.npy', allow_pickle=True).astype(np.float64)

kcwi_eric_628 = np.load('/home/habjan/SITELLE/sandbox_notebooks/sittelle_analysis/dig_subtraction/catalog_comp_arrays/SITELLE_KCWI_comp_eric_NGC0628.npy', allow_pickle=True).astype(np.float64)
kcwi_ryan_628 = np.load('/home/habjan/SITELLE/sandbox_notebooks/sittelle_analysis/dig_subtraction/catalog_comp_arrays/SITELLE_KCWI_comp_ryan_NGC0628.npy', allow_pickle=True).astype(np.float64)
kcwi_ebv_628 = np.load('/home/habjan/SITELLE/sandbox_notebooks/sittelle_analysis/dig_subtraction/catalog_comp_arrays/SITELLE_KCWI_comp_ebv_NGC0628.npy', allow_pickle=True).astype(np.float64)

kcwi_eric_2835 = np.load('/home/habjan/SITELLE/sandbox_notebooks/sittelle_analysis/dig_subtraction/catalog_comp_arrays/SITELLE_KCWI_comp_eric_NGC2835.npy', allow_pickle=True).astype(np.float64)
kcwi_ryan_2835 = np.load('/home/habjan/SITELLE/sandbox_notebooks/sittelle_analysis/dig_subtraction/catalog_comp_arrays/SITELLE_KCWI_comp_ryan_NGC2835.npy', allow_pickle=True).astype(np.float64)
kcwi_ebv_2835 = np.load('/home/habjan/SITELLE/sandbox_notebooks/sittelle_analysis/dig_subtraction/catalog_comp_arrays/SITELLE_KCWI_comp_ebv_NGC2835.npy', allow_pickle=True).astype(np.float64)

### Mask data

In [ ]:
def good_mask(x, y, c):
    return np.isfinite(x) & np.isfinite(y) & np.isfinite(c) & (x > 0) & (y > 0)

sig_mask = good_mask(sig_laurie_oii, sig_eric_oii, sig_ebv)

kcwi_628_mask = good_mask(kcwi_ryan_628, kcwi_eric_628, kcwi_ebv_628)
kcwi_2835_mask = good_mask(kcwi_ryan_2835, kcwi_eric_2835, kcwi_ebv_2835)

all_ebv = np.concatenate([
    sig_ebv[sig_mask],
    kcwi_ebv_628[kcwi_628_mask],
    kcwi_ebv_2835[kcwi_2835_mask]
])
norm = Normalize(vmin=np.nanmin(all_ebv), vmax=np.nanmax(all_ebv))

all_data = np.concatenate([
    sig_eric_oii[sig_mask],
    kcwi_eric_628[kcwi_628_mask],
    kcwi_eric_2835[kcwi_2835_mask],
    sig_laurie_oii[sig_mask],
    kcwi_ryan_628[kcwi_628_mask],
    kcwi_ryan_2835[kcwi_2835_mask]
])
ymin = np.nanmin(all_data) / 2.5
ymax = np.nanmax(all_data) * 2.5

xmin_left, xmax_left = np.nanmin(all_data) / 2.5, np.nanmax(all_data) * 2.5
xmin_right, xmax_right = np.nanmin(all_data) / 2.5, np.nanmax(all_data) * 2.5

### Make plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True, constrained_layout=True)

label_kwargs = dict(fontsize=12)

# ----------------------------
# Left panel: SIGNALS
# ----------------------------
ax = axes[0]
sc1 = ax.scatter(
    sig_laurie_oii[sig_mask],
    sig_eric_oii[sig_mask],
    c=sig_ebv[sig_mask],
    cmap='viridis',
    norm=norm,
    marker='o',
    s=35,
    label='NGC 628'
)

one_one_left = np.logspace(np.log10(max(xmin_left, ymin)), np.log10(min(xmax_left, ymax)), 200)
ax.plot(one_one_left, one_one_left, c='k', lw=1)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(xmin_left, xmax_left)
ax.set_ylim(ymin, ymax)
ax.set_xlabel(r'SIGNALS [OII] Flux [$10^{-20}$ erg cm$^{-2}$ s$^{-1}$]', **label_kwargs)
ax.set_ylabel(r'This work [OII] Flux [$10^{-20}$ erg cm$^{-2}$ s$^{-1}$]', **label_kwargs)
ax.legend()

# ----------------------------
# Right panel: KCWI
# ----------------------------
ax = axes[1]
sc2 = ax.scatter(
    kcwi_ryan_628[kcwi_628_mask],
    kcwi_eric_628[kcwi_628_mask],
    c=kcwi_ebv_628[kcwi_628_mask],
    cmap='viridis',
    norm=norm,
    marker='o',
    s=35,
    label='NGC 628'
)

ax.scatter(
    kcwi_ryan_2835[kcwi_2835_mask],
    kcwi_eric_2835[kcwi_2835_mask],
    c=kcwi_ebv_2835[kcwi_2835_mask],
    cmap='viridis',
    norm=norm,
    marker='^',
    s=45,
    label='NGC 2835'
)

one_one_right = np.logspace(np.log10(max(xmin_right, ymin)), np.log10(min(xmax_right, ymax)), 200)
ax.plot(one_one_right, one_one_right, c='k', lw=1)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(xmin_right, xmax_right)
ax.set_ylim(ymin, ymax)
ax.set_xlabel(r'KCWI [OII] Flux [$10^{-20}$ erg cm$^{-2}$ s$^{-1}$]', **label_kwargs)
ax.set_ylabel('')
ax.legend()

# ----------------------------
# Shared colorbar
# ----------------------------
cbar = fig.colorbar(sc2, ax=axes, shrink=0.95)
cbar.set_label(r'$E(B-V)$', fontsize=14, fontweight='semibold')

fig.savefig(path_out+'lit_validation_plot.png', dpi=300,bbox_inches='tight')

# Figure 11: 2D histograms comparing eight metallicity calibrations

In [ ]:
# figure

fig, axs = plt.subplots(7,7,figsize=(14,12),gridspec_kw={'hspace': 0, 'wspace': 0})
axsF = axs.flat

for ax in axsF:
    ax.set_xticks([7.0,8.0,8.5,9.0])
    ax.set_yticks([7.0,8.0,8.5,9.0])


c = plot_one(v1, v2,axs[0,0])
axs[0,0].set_xticklabels([])
axs[0,0].set_ylabel(v2_label,size=16, rotation=0, labelpad=40)
plot_one(v1, v3,axs[1,0])
axs[1,0].set_xticklabels([])
axs[1,0].set_ylabel(v3_label,size=16, rotation=0, labelpad=40)
plot_one(v1, v4,axs[2,0])
axs[2,0].set_xticklabels([])
axs[2,0].set_ylabel(v4_label,size=16, rotation=0, labelpad=40)
plot_one(v1, v5,axs[3,0])
axs[3,0].set_xticklabels([])
axs[3,0].set_ylabel(v5_label,size=16, rotation=0, labelpad=40)
plot_one(v1, v6,axs[4,0])
axs[4,0].set_xticklabels([])
axs[4,0].set_ylabel(v6_label,size=16, rotation=0, labelpad=40)
plot_one(v1, v7,axs[5,0])
axs[5,0].set_xticklabels([])
axs[5,0].set_ylabel(v7_label,size=16,rotation=0, labelpad=40)
plot_one(v1, v8,axs[6,0])
axs[6,0].set_ylabel(v8_label,size=16, rotation=0, labelpad=40)
axs[6,0].set_xlabel(v1_label,size=16)

fig.delaxes(axs[0,1])
plot_one(v2, v3,axs[1,1])
axs[1,1].set_xticklabels([])
axs[1,1].set_yticklabels([])
plot_one(v2, v4,axs[2,1])
axs[2,1].set_xticklabels([])
axs[2,1].set_yticklabels([])
plot_one(v2, v5,axs[3,1])
axs[3,1].set_xticklabels([])
axs[3,1].set_yticklabels([])
plot_one(v2, v6,axs[4,1])
axs[4,1].set_xticklabels([])
axs[4,1].set_yticklabels([])
plot_one(v2, v7,axs[5,1])
axs[5,1].set_xticklabels([])
axs[5,1].set_yticklabels([])
plot_one(v2, v8,axs[6,1])
axs[6,1].set_yticklabels([])
axs[6,1].set_xlabel(v2_label,size=16)

fig.delaxes(axs[0,2])
fig.delaxes(axs[1,2])
plot_one(v3, v4,axs[2,2])
axs[2,2].set_xticklabels([])
axs[2,2].set_yticklabels([])
plot_one(v3, v5,axs[3,2])
axs[3,2].set_xticklabels([])
axs[3,2].set_yticklabels([])
plot_one(v3, v6,axs[4,2])
axs[4,2].set_xticklabels([])
axs[4,2].set_yticklabels([])
plot_one(v3, v7,axs[5,2])
axs[5,2].set_xticklabels([])
axs[5,2].set_yticklabels([])
plot_one(v3, v8,axs[6,2])
axs[6,2].set_yticklabels([])
axs[6,2].set_xlabel(v3_label,size=16)

fig.delaxes(axs[0,3])
fig.delaxes(axs[1,3])
fig.delaxes(axs[2,3])
plot_one(v4, v5,axs[3,3])
axs[3,3].set_xticklabels([])
axs[3,3].set_yticklabels([])
plot_one(v4, v6,axs[4,3])
axs[4,3].set_xticklabels([])
axs[4,3].set_yticklabels([])
plot_one(v4, v7,axs[5,3])
axs[5,3].set_xticklabels([])
axs[5,3].set_yticklabels([])
plot_one(v4, v8,axs[6,3])
axs[6,3].set_yticklabels([])
axs[6,3].set_xlabel(v4_label,size=16)

fig.delaxes(axs[0,4])
fig.delaxes(axs[1,4])
fig.delaxes(axs[2,4])
fig.delaxes(axs[3,4])
plot_one(v5, v6,axs[4,4])
axs[4,4].set_xticklabels([])
axs[4,4].set_yticklabels([])
plot_one(v5, v7,axs[5,4])
axs[5,4].set_xticklabels([])
axs[5,4].set_yticklabels([])
plot_one(v5, v8,axs[6,4])
axs[6,4].set_yticklabels([])
axs[6,4].set_xlabel(v5_label,size=16)

fig.delaxes(axs[0,5])
fig.delaxes(axs[1,5])
fig.delaxes(axs[2,5])
fig.delaxes(axs[3,5])
fig.delaxes(axs[4,5])
plot_one(v6, v7,axs[5,5])
axs[5,5].set_xticklabels([])
axs[5,5].set_yticklabels([])
plot_one(v6, v8,axs[6,5])
axs[6,5].set_yticklabels([])
axs[6,5].set_xlabel(v6_label,size=16)

fig.delaxes(axs[0,6])
fig.delaxes(axs[1,6])
fig.delaxes(axs[2,6])
fig.delaxes(axs[3,6])
fig.delaxes(axs[4,6])
fig.delaxes(axs[5,6])
c = plot_one(v7, v8,axs[6,6])
axs[6,6].set_yticklabels([])
axs[6,6].set_xlabel(v7_label,size=16)




# add color bars
cbaxes = fig.add_axes([0.7, 0.48, 0.15, 0.02])
cb = plt.colorbar(c,cax = cbaxes,orientation='horizontal',ticks=[0,1.0,2.0,3.0,4.0])
cb.set_label('log(N)')

labelx=fig.text(0.5, 0.03, r'12+log(O/H)', ha='center',size=20)
labely=fig.text(-0.04, 0.5, r'12+log(O/H)', va='center', rotation='vertical',size=20)

fig.savefig(path_out+'Figure_corner_plot.png', dpi=300,bbox_inches='tight')
